# QAOA for weighted Max-Cut

This notebook solves the **Quantum Optimization of Network Partitioning** challenge on a six-vertex weighted graph. It includes an exact classical baseline, reproducible p=1 QAOA, shot-based results, and a hardware-topology comparison.

> **Input status:** the challenge prompt contained placeholders rather than the locked Phase 2 graph and Processor A/B/D specifications. The editable configuration below therefore uses clearly labeled *example* inputs and provisional 6+-qubit topologies, as authorized. Replace those values with the official inputs when available. The analysis and plots are generated from whichever inputs are configured.

In [ ]:
%pip install qiskit qiskit-aer scipy networkx matplotlib numpy

## Configuration and input loading

Edit this single configuration cell to replace the example graph, processor coupling maps, native gates, noise assumptions, or seeds. If `instances.json` exists next to this notebook, its graph data takes precedence over the fallback edge list. All configured example edge weights are integers from 1 through 5.

In [ ]:
from pathlib import Path
import json
import random

import numpy as np

# Reproducibility: every stochastic step uses one of these fixed seeds.
SEED = 20261029
OPTIMIZER_SEED = 20261030
SAMPLER_SEED = 20261031
TRANSPILER_SEED = 20261032
NOISE_SEED = 20261033
SHOTS = 8192
N_VERTICES = 6
N_RESTARTS = 20

# EXAMPLE ONLY: replace with the locked Phase 2 graph when it is supplied.
EXAMPLE_EDGE_LIST = [
    (0, 1, 3), (0, 2, 1), (0, 5, 2),
    (1, 2, 4), (1, 3, 2),
    (2, 3, 5), (2, 4, 1),
    (3, 4, 3), (3, 5, 2),
    (4, 5, 4),
]

# PROVISIONAL EXAMPLE TOPOLOGIES ONLY. All have at least six qubits.
# Coupling links are supplied as undirected pairs; the code expands directions.
# Native basis is intentionally consistent across processors for fair comparison.
NATIVE_BASIS_GATES = ["rz", "sx", "x", "cx"]
PROCESSOR_CONFIG = {
    "A": {
        "name": "Processor A (provisional linear)",
        "num_qubits": 6,
        "coupling_map": [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5)],
        "basis_gates": NATIVE_BASIS_GATES,
        "noise": {"depolarizing_1q": 0.001, "depolarizing_2q": 0.010, "readout": 0.020},
    },
    "B": {
        "name": "Processor B (provisional ring)",
        "num_qubits": 8,
        "coupling_map": [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7), (7, 0)],
        "basis_gates": NATIVE_BASIS_GATES,
        "noise": {"depolarizing_1q": 0.001, "depolarizing_2q": 0.015, "readout": 0.025},
    },
    "D": {
        "name": "Processor D (provisional all-to-all)",
        "num_qubits": 6,
        "coupling_map": [(u, v) for u in range(6) for v in range(u + 1, 6)],
        "basis_gates": NATIVE_BASIS_GATES,
        "noise": {"depolarizing_1q": 0.001, "depolarizing_2q": 0.008, "readout": 0.015},
    },
}

def extract_edges(payload):
    """Accept common instance JSON edge-list shapes without changing edge semantics."""
    if isinstance(payload, list):
        return payload
    if isinstance(payload, dict):
        for key in ("edges", "edge_list", "graph"):
            if key in payload:
                candidate = payload[key]
                if isinstance(candidate, dict) and "edges" in candidate:
                    return candidate["edges"]
                return candidate
    raise ValueError("instances.json must contain an edge list or an 'edges'/'edge_list' field.")

instance_candidates = [Path("instances.json"), Path("Phase 2/instances.json")]
instance_path = next((path for path in instance_candidates if path.exists()), Path("instances.json"))
if instance_path.exists():
    raw_payload = json.loads(instance_path.read_text(encoding="utf-8"))
    raw_edges = extract_edges(raw_payload)
    print(f"Loaded graph from {instance_path.resolve()}")
else:
    raw_edges = EXAMPLE_EDGE_LIST
    print("instances.json not found; using the clearly labeled example edge list.")

EDGE_LIST = [tuple(edge) for edge in raw_edges]
assert len(EDGE_LIST) > 0, "The graph must contain at least one edge."
assert all(len(edge) == 3 for edge in EDGE_LIST), "Each edge must be (u, v, weight)."
assert all(isinstance(u, int) and isinstance(v, int) for u, v, _ in EDGE_LIST)
assert all(isinstance(w, int) and 1 <= w <= 5 for _, _, w in EDGE_LIST), "Weights must be integers 1-5."
assert all(0 <= u < N_VERTICES and 0 <= v < N_VERTICES and u != v for u, v, _ in EDGE_LIST)
assert len({(min(u, v), max(u, v)) for u, v, _ in EDGE_LIST}) == len(EDGE_LIST), "Duplicate undirected edges are not supported."
for key, spec in PROCESSOR_CONFIG.items():
    assert spec["num_qubits"] >= N_VERTICES, f"Processor {key} needs at least six qubits."
    assert all(0 <= u < spec["num_qubits"] and 0 <= v < spec["num_qubits"] for u, v in spec["coupling_map"])

random.seed(SEED)
np.random.seed(SEED)
print(f"Graph: {N_VERTICES} vertices, {len(EDGE_LIST)} weighted edges")
print("Processor inputs are provisional examples; replace them with official locked specifications.")

## 1. Mathematical formulation

For a binary assignment $x_i\in\{0,1\}$, vertices in different groups have unequal bits. The weighted cut is

$$
C(x)=\sum_{(i,j)\in E} w_{ij}\,\mathbf{1}[x_i\ne x_j]
    =\sum_{(i,j)\in E}w_{ij}(x_i+x_j-2x_ix_j),
\qquad \max_{x\in\{0,1\}^6} C(x).
$$

This is a QUBO maximization. With $x_i=(1-z_i)/2$ and $z_i\in\{-1,+1\}$, unequal bits correspond to $z_i z_j=-1$. Under the **positive-cut sign convention**, the cost Hamiltonian is

$$
H_C=\sum_{(i,j)\in E}\frac{w_{ij}}{2}(I-Z_iZ_j),
$$

so a computational-basis measurement has eigenvalue equal to its cut weight (not its negative). The standard transverse-field mixer is $H_B=\sum_i X_i$. The p-layer QAOA alternates cost and mixer evolutions.

## 2. QUBO construction and exhaustive verification

For a symmetric matrix $Q$, binary variables satisfy $x_i^2=x_i$, so $x^TQx$ includes both symmetric off-diagonal entries. We set $Q_{ii}$ to the weighted degree and $Q_{ij}=Q_{ji}=-w_{ij}$ for each edge. This makes $x^TQx$ the **maximized cut weight**, with no sign flip needed.

In [ ]:
from itertools import product

import networkx as nx

G = nx.Graph()
G.add_nodes_from(range(N_VERTICES))
G.add_weighted_edges_from(EDGE_LIST)

Q = np.zeros((N_VERTICES, N_VERTICES), dtype=float)
for u, v, weight in EDGE_LIST:
    Q[u, u] += weight
    Q[v, v] += weight
    Q[u, v] -= weight
    Q[v, u] -= weight

ALL_BITS = list(product((0, 1), repeat=N_VERTICES))

def cut_weight(bits):
    """Return the weighted cut value for bits ordered by vertex index."""
    return sum(weight for u, v, weight in EDGE_LIST if bits[u] != bits[v])

for bits in ALL_BITS:
    x = np.asarray(bits, dtype=float)
    assert np.isclose(x @ Q @ x, cut_weight(bits)), f"QUBO mismatch for {bits}"

print("QUBO matrix Q (x.T @ Q @ x maximizes the cut):")
print(Q)
print(f"Verified x.T @ Q @ x against the cut function for all {len(ALL_BITS)} bitstrings.")

## 3. Classical exact optimum

Brute force enumerates all $2^6=64$ assignments. Since swapping the two partition labels complements every bit, every optimum has a complementary optimum.

In [ ]:
def bits_to_string(bits):
    return "".join(str(int(bit)) for bit in bits)

cut_values = {bits: cut_weight(bits) for bits in ALL_BITS}
classical_optimum = max(cut_values.values())
optimal_bits = sorted(bits for bits, value in cut_values.items() if value == classical_optimum)
optimal_bitstrings = [bits_to_string(bits) for bits in optimal_bits]

assert len(cut_values) == 2 ** N_VERTICES
assert all(bits_to_string(tuple(1 - bit for bit in bits)) in optimal_bitstrings for bits in optimal_bits)
print(f"Classical maximum cut: {classical_optimum}")
print(f"All optimal bitstrings (vertex order 0..5): {optimal_bitstrings}")
print(f"Number of optimal assignments: {len(optimal_bitstrings)}")

## 4. Manual p=1 QAOA circuit and exact expectation

Each state begins in $|+\rangle^{\otimes 6}$, applies the requested per-edge `RZZ(2*gamma*weight)` cost layer and `RX(2*beta)` mixer, then optionally measures. The explicit `2*gamma*weight` angle is retained as required; its scale is absorbed into the optimized gamma parameter. Optimization uses exact statevector expectations, not finite-shot estimates.

In [ ]:
from scipy.optimize import minimize
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
import pandas as pd

def build_qaoa_circuit(gamma, beta, measure=False):
    circuit = QuantumCircuit(N_VERTICES, N_VERTICES if measure else 0)
    circuit.h(range(N_VERTICES))
    for u, v, weight in EDGE_LIST:
        circuit.rzz(2.0 * gamma * weight, u, v)
    for qubit in range(N_VERTICES):
        circuit.rx(2.0 * beta, qubit)
    if measure:
        circuit.measure(range(N_VERTICES), range(N_VERTICES))
    return circuit

def index_to_bits(index):
    # Qiskit statevector index uses qubit 0 as the least significant bit.
    return tuple((index >> qubit) & 1 for qubit in range(N_VERTICES))

def exact_expected_cut(gamma, beta):
    state = Statevector.from_instruction(build_qaoa_circuit(gamma, beta))
    probabilities = state.probabilities()
    return float(sum(probabilities[index] * cut_weight(index_to_bits(index))
                     for index in range(len(probabilities))))

# Sanity check: the exact expectation at p=1 is a probability-weighted
# enumeration of the same cut function used by the QUBO and brute-force baseline.
test_gamma, test_beta = 0.37, 0.29
test_state = Statevector.from_instruction(build_qaoa_circuit(test_gamma, test_beta))
enumerated_expectation = sum(
    test_state.probabilities()[index] * cut_weight(index_to_bits(index))
    for index in range(2 ** N_VERTICES)
)
assert np.isclose(exact_expected_cut(test_gamma, test_beta), enumerated_expectation)
assert exact_expected_cut(test_gamma, test_beta) <= classical_optimum + 1e-10
print("Statevector expectation agrees with explicit brute-force probability weighting.")

## 5. Twenty reproducible COBYLA restarts

Each run begins from an independently seeded random $(\gamma,\beta)$ pair. The table reports the best exact expected cut reached by each optimization; mean, population standard deviation, and best are summarized below.

In [ ]:
restart_rng = np.random.default_rng(OPTIMIZER_SEED)
restart_rows = []

for restart in range(N_RESTARTS):
    initial = np.array([
        restart_rng.uniform(0.0, 2.0 * np.pi),
        restart_rng.uniform(0.0, np.pi),
    ])
    result = minimize(
        lambda params: -exact_expected_cut(params[0], params[1]),
        initial,
        method="COBYLA",
        options={"maxiter": 120, "rhobeg": 0.5, "tol": 1e-5},
    )
    restart_rows.append({
        "restart": restart + 1,
        "initial_gamma": float(initial[0]),
        "initial_beta": float(initial[1]),
        "gamma": float(result.x[0]),
        "beta": float(result.x[1]),
        "expected_cut": float(-result.fun),
        "success": bool(result.success),
        "evaluations": int(result.nfev),
    })

restart_table = pd.DataFrame(restart_rows)
restart_mean = restart_table["expected_cut"].mean()
restart_std = restart_table["expected_cut"].std(ddof=0)
best_restart = restart_table.loc[restart_table["expected_cut"].idxmax()]
best_gamma = float(best_restart["gamma"])
best_beta = float(best_restart["beta"])
best_expected_cut = float(best_restart["expected_cut"])

display(restart_table)
print(f"Expected cut over {N_RESTARTS} restarts: mean={restart_mean:.6f}, "
      f"std={restart_std:.6f}, best={best_expected_cut:.6f}")
assert best_expected_cut <= classical_optimum + 1e-8

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(restart_table["expected_cut"], bins=10, color="#4C78A8", edgecolor="white")
ax.axvline(restart_mean, color="#F58518", linestyle="--", label=f"Mean {restart_mean:.3f}")
ax.axvline(best_expected_cut, color="#54A24B", linestyle=":", label=f"Best {best_expected_cut:.3f}")
ax.set(title="p=1 QAOA restart distribution", xlabel="Final exact expected cut", ylabel="Restart count")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Final ideal shot-based results

The best parameter pair by exact expected cut is sampled with a fixed seed and 8,192 shots. Count keys are converted from Qiskit's displayed classical-bit order to vertex order 0 through 5.

In [ ]:
def counts_to_vertex_order(counts):
    return {key.replace(" ", "")[::-1]: value for key, value in counts.items()}

ideal_circuit = build_qaoa_circuit(best_gamma, best_beta, measure=True)
ideal_backend = AerSimulator()
ideal_result = ideal_backend.run(
    ideal_circuit, shots=SHOTS, seed_simulator=SAMPLER_SEED
).result()
ideal_counts = counts_to_vertex_order(ideal_result.get_counts())
ideal_shot_values = {
    bitstring: cut_weight(tuple(int(bit) for bit in bitstring))
    for bitstring in ideal_counts
}
best_sampled_cut = max(ideal_shot_values.values())
sampled_best_strings = sorted(
    bitstring for bitstring, value in ideal_shot_values.items()
    if value == best_sampled_cut
)
ideal_optimal_probability = sum(
    count for bitstring, count in ideal_counts.items() if bitstring in optimal_bitstrings
) / SHOTS
sampled_expected_cut = sum(
    ideal_counts[bitstring] * ideal_shot_values[bitstring] for bitstring in ideal_counts
) / SHOTS
best_approximation_ratio = best_sampled_cut / classical_optimum
expected_approximation_ratio = sampled_expected_cut / classical_optimum

print(f"Optimized parameters: gamma={best_gamma:.6f}, beta={best_beta:.6f}")
print(f"Best sampled cut: {best_sampled_cut} (bitstring(s): {sampled_best_strings})")
print(f"Best-cut approximation ratio: {best_approximation_ratio:.6f}")
print(f"Shot-based expected cut: {sampled_expected_cut:.6f}")
print(f"Expected-value approximation ratio: {expected_approximation_ratio:.6f}")
print(f"Probability of sampling any optimum: {ideal_optimal_probability:.6f}")
assert 0.0 <= ideal_optimal_probability <= 1.0

In [ ]:
all_bitstrings = [bits_to_string(bits) for bits in ALL_BITS]
plot_counts = [ideal_counts.get(bitstring, 0) for bitstring in all_bitstrings]
bar_colors = ["#E45756" if bitstring in optimal_bitstrings else "#4C78A8"
              for bitstring in all_bitstrings]

fig, ax = plt.subplots(figsize=(15, 5))
ax.bar(all_bitstrings, plot_counts, color=bar_colors)
ax.set(title="Ideal p=1 sampled bitstring distribution (red = classical optimum)",
       xlabel="Bitstring in vertex order 0..5", ylabel="Shots")
ax.tick_params(axis="x", rotation=90)
plt.tight_layout()
plt.show()

## 7. Graph visualization

Nodes are colored by the best sampled partition. Dashed red edges cross the partition; solid gray edges remain within one group. Edge labels show the configured weights.

In [ ]:
import matplotlib.pyplot as plt

best_partition_string = sampled_best_strings[0]
best_partition = tuple(int(bit) for bit in best_partition_string)
node_colors = ["#4C78A8" if best_partition[node] == 0 else "#F58518"
               for node in range(N_VERTICES)]
edge_colors = [
    "#E45756" if best_partition[u] != best_partition[v] else "#9A9A9A"
    for u, v, _ in EDGE_LIST
]
edge_widths = [
    1.5 + 0.5 * weight if best_partition[u] != best_partition[v] else 1.0
    for u, v, weight in EDGE_LIST
]
positions = nx.spring_layout(G, seed=SEED)
fig, ax = plt.subplots(figsize=(7, 6))
nx.draw_networkx_nodes(G, positions, node_color=node_colors, node_size=850, ax=ax)
nx.draw_networkx_labels(G, positions, font_color="white", font_weight="bold", ax=ax)
nx.draw_networkx_edges(G, positions, edge_color=edge_colors, width=edge_widths, ax=ax)
nx.draw_networkx_edge_labels(
    G, positions, edge_labels=nx.get_edge_attributes(G, "weight"), ax=ax
)
ax.set_title(f"Best sampled partition {best_partition_string}; red edges are cut")
ax.axis("off")
plt.tight_layout()
plt.show()

## 8. Hardware-aware transpilation and noisy simulation

The same optimized measured p=1 circuit is compiled for all three configured processors with the same basis gates, optimization level, and transpiler seed. The unconstrained compilation gives the logical CX baseline; constrained-minus-logical CX is the routing overhead. The reported swap count is the number of explicit SWAP instructions remaining in the transpiled circuit (basis decomposition may instead express routing as extra CX gates).

Noisy Aer runs use the configured per-processor depolarizing and readout assumptions. These noise values and all three topologies are provisional examples, not measured hardware properties.

In [ ]:
from qiskit.transpiler import CouplingMap
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError

logical_circuit = build_qaoa_circuit(best_gamma, best_beta, measure=True)
common_basis = PROCESSOR_CONFIG["A"]["basis_gates"]
logical_compiled = transpile(
    logical_circuit,
    basis_gates=common_basis,
    optimization_level=1,
    seed_transpiler=TRANSPILER_SEED,
)
logical_cx_count = logical_compiled.count_ops().get("cx", 0)

def make_noise_model(noise_spec):
    model = NoiseModel()
    p1 = noise_spec["depolarizing_1q"]
    p2 = noise_spec["depolarizing_2q"]
    readout = noise_spec["readout"]
    model.add_all_qubit_quantum_error(depolarizing_error(p1, 1), ["rz", "sx", "x"])
    model.add_all_qubit_quantum_error(depolarizing_error(p2, 2), ["cx"])
    model.add_all_qubit_readout_error(ReadoutError([[1 - readout, readout],
                                                    [readout, 1 - readout]]))
    return model

hardware_rows = []
hardware_counts = {}
for processor_key, spec in PROCESSOR_CONFIG.items():
    directed_links = [link for u, v in spec["coupling_map"] for link in ((u, v), (v, u))]
    coupling = CouplingMap(directed_links)
    compiled = transpile(
        logical_circuit,
        coupling_map=coupling,
        basis_gates=spec["basis_gates"],
        initial_layout=list(range(N_VERTICES)),
        optimization_level=1,
        seed_transpiler=TRANSPILER_SEED,
    )
    operations = compiled.count_ops()
    cx_count = int(operations.get("cx", 0))
    swap_count = int(operations.get("swap", 0))
    layout = compiled.layout
    if layout is None:
        final_layout = list(range(N_VERTICES))
    else:
        final_layout = layout.final_index_layout()[:N_VERTICES]

    noisy_backend = AerSimulator(noise_model=make_noise_model(spec["noise"]))
    noisy_result = noisy_backend.run(
        compiled, shots=SHOTS, seed_simulator=NOISE_SEED
    ).result()
    noisy_counts = counts_to_vertex_order(noisy_result.get_counts())
    hardware_counts[processor_key] = noisy_counts
    noisy_optimal_probability = sum(
        count for bitstring, count in noisy_counts.items()
        if bitstring in optimal_bitstrings
    ) / SHOTS

    hardware_rows.append({
        "processor": processor_key,
        "name": spec["name"],
        "qubits": spec["num_qubits"],
        "depth": compiled.depth(),
        "CX_count": cx_count,
        "logical_CX_baseline": int(logical_cx_count),
        "routing_CX_overhead": int(cx_count - logical_cx_count),
        "explicit_SWAP_count": swap_count,
        "final_logical_to_physical_layout": str(final_layout),
        "noisy_optimal_probability": noisy_optimal_probability,
        "basis_gates": ",".join(spec["basis_gates"]),
        "optimization_level": 1,
        "seed_transpiler": TRANSPILER_SEED,
    })

hardware_table = pd.DataFrame(hardware_rows)
display(hardware_table)
assert len(hardware_table) == 3
assert (hardware_table["CX_count"] >= 0).all()

## 9. Discussion, comparison, and limitations

Connectivity affects compilation because each QAOA cost interaction must be implemented between the corresponding logical qubits. Sparse topologies can require routing, increasing depth and two-qubit-gate count; those gates are often a major source of noise and can lower the probability of measuring an optimum. Higher degree and smaller diameter generally reduce routing, while all-to-all connectivity can implement every edge interaction directly (subject to gate scheduling and native-gate decomposition).

The comparison below is generated from this run's measured compilation and simulation results. With these **provisional** assumptions, the recommended topology is the one with the highest noisy optimal-state probability; CX overhead and depth provide additional context. This is a simulator-based recommendation, not a claim about actual processors.

Limitations: the graph and hardware specifications are example inputs because the official Phase 2 values were absent; the p=1 ansatz can have a nonzero optimality gap; COBYLA results depend on initial points and iteration budget; the noise model is simplified and omits calibration variation, crosstalk, and drift; transpiled depth and noisy success are backend/model dependent. The exact classical optimum remains the reference for this six-vertex instance.

In [ ]:
recommended = hardware_table.sort_values(
    ["noisy_optimal_probability", "routing_CX_overhead", "depth"],
    ascending=[False, True, True],
).iloc[0]
print(
    f"Best noisy optimal-state probability in this run: {recommended['name']} "
    f"({recommended['noisy_optimal_probability']:.6f})."
)
print(
    "Recommendation is conditional on the provisional graph, topology, and noise "
    "assumptions in the configuration cell."
)

## 10. Summary metrics and CSV export

The final table collects the exact baseline, ideal p=1 result, restart statistics, and per-processor compilation/noisy-sampling metrics. It is exported as `results.csv` in the notebook's current working directory.

In [ ]:
summary_rows = [{
    "category": "classical_exact",
    "processor": "reference",
    "classical_optimum": classical_optimum,
    "best_sampled_cut": np.nan,
    "best_cut_approximation_ratio": np.nan,
    "expected_cut": np.nan,
    "expected_approximation_ratio": np.nan,
    "optimal_sampling_probability": np.nan,
    "restart_mean_expected_cut": np.nan,
    "restart_std_expected_cut": np.nan,
    "circuit_depth": np.nan,
    "CX_count": np.nan,
    "routing_CX_overhead": np.nan,
    "explicit_SWAP_count": np.nan,
    "final_layout": "",
}, {
    "category": "ideal_p1",
    "processor": "logical",
    "classical_optimum": classical_optimum,
    "best_sampled_cut": best_sampled_cut,
    "best_cut_approximation_ratio": best_approximation_ratio,
    "expected_cut": sampled_expected_cut,
    "expected_approximation_ratio": expected_approximation_ratio,
    "optimal_sampling_probability": ideal_optimal_probability,
    "restart_mean_expected_cut": restart_mean,
    "restart_std_expected_cut": restart_std,
    "circuit_depth": logical_compiled.depth(),
    "CX_count": int(logical_cx_count),
    "routing_CX_overhead": 0,
    "explicit_SWAP_count": int(logical_compiled.count_ops().get("swap", 0)),
    "final_layout": "logical",
}]

for row in hardware_rows:
    summary_rows.append({
        "category": "noisy_hardware_simulation",
        "processor": row["name"],
        "classical_optimum": classical_optimum,
        "best_sampled_cut": np.nan,
        "best_cut_approximation_ratio": np.nan,
        "expected_cut": np.nan,
        "expected_approximation_ratio": np.nan,
        "optimal_sampling_probability": row["noisy_optimal_probability"],
        "restart_mean_expected_cut": restart_mean,
        "restart_std_expected_cut": restart_std,
        "circuit_depth": row["depth"],
        "CX_count": row["CX_count"],
        "routing_CX_overhead": row["routing_CX_overhead"],
        "explicit_SWAP_count": row["explicit_SWAP_count"],
        "final_layout": row["final_logical_to_physical_layout"],
    })

summary_table = pd.DataFrame(summary_rows)
display(summary_table)
phase2_directory = Path("Phase 2") if Path("Phase 2").is_dir() else Path(".")
output_path = phase2_directory / "results.csv"
summary_table.to_csv(output_path, index=False)
print(f"Saved summary metrics to {output_path.resolve()}")